---

## **積立投資シミュレーター**

---

積立投資とは、毎月一定額をコツコツと投資し続ける方法です。

「現在価値と将来価値」で学んだ複利の考え方を使うと、積立投資でどのくらい資産が増えるかを計算できます。

このノートブックでは以下の3つを見ていきます：

1. **積立投資の将来価値** — 毎月いくら積み立てると、何年後にいくらになるか
2. **元本 vs 利益の内訳** — 積み立てたお金と運用益の比率
3. **FIRE達成シミュレーション** — 経済的自立に必要な資産と達成までの期間

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
sns.set(style="darkgrid")
import warnings
warnings.simplefilter('ignore')

# 日本語フォント設定（環境によっては変更が必要）
plt.rcParams['font.family'] = 'IPAexGothic'
import subprocess, sys
try:
    plt.rcParams['font.family'] = 'IPAexGothic'
    fig, ax = plt.subplots()
    ax.set_title('テスト')
    plt.close()
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

---

## **1. 積立投資の将来価値を計算する**

---

毎月一定額 $m$ を年利 $r$ で積み立てたとき、$n$ ヶ月後の将来価値は次の式で求められます：

$$FV = m \times \frac{(1 + r_{月})^n - 1}{r_{月}}$$

ここで $r_{月} = r / 12$（月利）です。

たとえば、毎月3万円を年利5%で30年間積み立てるといくらになるでしょう？

In [ ]:
def calc_future_value(monthly: float, annual_rate: float, years: int) -> float:
    """積立投資の将来価値を計算する"""
    r = annual_rate / 12
    n = years * 12
    if r == 0:
        return monthly * n
    return monthly * ((1 + r) ** n - 1) / r

monthly = 30000    # 毎月の積立額（円）
annual_rate = 0.05 # 年利（5%）
years = 30         # 積立期間（年）

fv = calc_future_value(monthly, annual_rate, years)
principal = monthly * years * 12

print(f"毎月 {monthly:,}円 を年利 {annual_rate*100:.1f}% で {years}年間 積み立てると...")
print(f"  元本合計:  {principal:>12,.0f} 円")
print(f"  将来価値:  {fv:>12,.0f} 円")
print(f"  運用益:    {fv - principal:>12,.0f} 円  （元本の {(fv/principal - 1)*100:.1f}% 増）")

---

## **2. 資産成長の推移をグラフで見る**

---

In [ ]:
def simulate_growth(monthly: float, annual_rate: float, years: int) -> pd.DataFrame:
    """年ごとの資産推移をDataFrameで返す"""
    records = []
    r = annual_rate / 12
    balance = 0.0
    for month in range(1, years * 12 + 1):
        balance = balance * (1 + r) + monthly
        if month % 12 == 0:
            year = month // 12
            principal = monthly * month
            records.append({
                "年数": year,
                "元本": principal,
                "運用益": balance - principal,
                "資産合計": balance,
            })
    return pd.DataFrame(records)

df = simulate_growth(monthly=30000, annual_rate=0.05, years=30)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 積み上げ棒グラフ
ax = axes[0]
ax.bar(df["年数"], df["元本"] / 1e6, label="元本", color="#4C72B0")
ax.bar(df["年数"], df["運用益"] / 1e6, bottom=df["元本"] / 1e6, label="運用益", color="#DD8452")
ax.set_xlabel("積立年数")
ax.set_ylabel("資産額（百万円）")
ax.set_title("元本 vs 運用益の推移")
ax.legend()

# 右: 折れ線グラフ（年利比較）
ax2 = axes[1]
for rate in [0.0, 0.03, 0.05, 0.07]:
    df2 = simulate_growth(monthly=30000, annual_rate=rate, years=30)
    ax2.plot(df2["年数"], df2["資産合計"] / 1e6, label=f"年利 {rate*100:.0f}%")
ax2.set_xlabel("積立年数")
ax2.set_ylabel("資産額（百万円）")
ax2.set_title("年利別 資産成長の比較（毎月3万円）")
ax2.legend()

plt.tight_layout()
plt.show()

print("\n30年後の資産額（毎月3万円）")
for rate in [0.0, 0.03, 0.05, 0.07]:
    fv = calc_future_value(30000, rate, 30)
    print(f"  年利 {rate*100:.0f}%: {fv:>12,.0f} 円")

---

## **3. FIREシミュレーション**

---

**FIRE（Financial Independence, Retire Early）** とは経済的自立・早期リタイアのことです。

FIREの基本ルール「4%ルール」によれば、**年間支出の25倍**の資産があれば、年4%ずつ取り崩しても資産が枯渇しないと言われています。

$$\text{FIRE目標額} = \text{年間支出} \times 25$$

あなたが毎月いくら積み立てれば、何年でFIREできるかを計算してみましょう。

In [ ]:
def years_to_fire(
    monthly_savings: float,
    monthly_expense: float,
    annual_rate: float,
    initial_assets: float = 0,
    max_years: int = 60
) -> int | None:
    """FIRE達成までの年数を返す（達成不可の場合はNone）"""
    fire_target = monthly_expense * 12 * 25
    r = annual_rate / 12
    balance = initial_assets
    for month in range(1, max_years * 12 + 1):
        balance = balance * (1 + r) + monthly_savings
        if balance >= fire_target:
            return month / 12
    return None

# パラメータ設定
monthly_expense = 200000  # 月の生活費（円）
annual_rate = 0.05        # 年利
initial_assets = 0        # 初期資産

fire_target = monthly_expense * 12 * 25
print(f"月の生活費: {monthly_expense:,}円")
print(f"FIRE目標額: {fire_target:,}円（{fire_target/1e8:.1f}億円）")
print()

print("積立額別 FIRE達成年数（年利5%）")
print("-" * 35)
for savings in [50000, 100000, 150000, 200000, 300000, 500000]:
    result = years_to_fire(savings, monthly_expense, annual_rate, initial_assets)
    if result:
        print(f"  毎月 {savings:>7,}円: {result:5.1f}年")
    else:
        print(f"  毎月 {savings:>7,}円: 60年以内に達成できず")

In [ ]:
# FIREシミュレーションのヒートマップ
savings_list = np.arange(50000, 550000, 50000)
expense_list = np.arange(100000, 550000, 50000)

matrix = []
for expense in expense_list:
    row = []
    for savings in savings_list:
        result = years_to_fire(savings, expense, 0.05, 0)
        row.append(result if result else 60)
    matrix.append(row)

df_heat = pd.DataFrame(
    matrix,
    index=[f"{int(e/1e4)}万" for e in expense_list],
    columns=[f"{int(s/1e4)}万" for s in savings_list]
)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    df_heat,
    annot=True, fmt=".0f",
    cmap="RdYlGn_r",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "FIRE達成年数"}
)
ax.set_xlabel("毎月の積立額")
ax.set_ylabel("毎月の生活費")
ax.set_title("FIRE達成年数ヒートマップ（年利5%）\n数値はFIRE達成までの年数（60=60年以内未達成）")
plt.tight_layout()
plt.show()

---

## **まとめ**

---

積立投資のポイントをまとめると：

- **早く始めるほど有利** — 複利の効果で、時間が長いほど運用益が大きく育つ
- **年利の差は大きい** — 年利0%と7%では30年後の資産額に何倍もの差が出る
- **FIREは生活費次第** — 節約して生活費を下げることが、積立額を増やすのと同じ効果を持つ

「現在価値と将来価値」で学んだ複利の魔法が、積立投資ではフル活用されています。

毎月コツコツ積み立てて、お金に働いてもらいましょう！